# Generate Storybook Stories for Evaluation (in-place)

In [19]:
import re
import json
from pathlib import Path

### 1. Repo root and configuration

In [20]:
def find_repo_root(marker: str = 'dataset', start: Path | None = None) -> Path:
    """Searches upward from the current working directory until a folder
    named 'marker' is found -- robust against the kernel's working directory
    not matching the notebook's own folder."""
    start = start or Path.cwd()

    for parent in [start, *start.parents]:
        if (parent / marker).is_dir():
            return parent

    raise FileNotFoundError(f"Could not find a folder named '{marker}' above {start}")


REPO_ROOT = find_repo_root()
STORYBOOK_ROOT = REPO_ROOT / 'dataset' / 'storybook'

# 'components' or 'uis'
TYPE = 'uis'

CODE_DIR = STORYBOOK_ROOT / 'src' / 'code' / TYPE
STORIES_DIR = STORYBOOK_ROOT / 'src' / 'stories' / TYPE

APPROACHES = ['b', 'c', 'd']
PROMPT_STRATEGIES = ['few_shot', 'zero_shot']
COMPLEXITIES = ['hard', 'medium', 'simple']

MANIFEST_PATH = REPO_ROOT / 'evaluations' / f'stories_manifest_{TYPE}.json'

BEGIN_MARKER = '// >>> AUTO-GENERATED EVAL STORIES (managed by storybook-generate-stories.ipynb) -- do not edit by hand >>>'
END_MARKER = '// <<< AUTO-GENERATED EVAL STORIES <<<'

print(f'CODE_DIR:    {CODE_DIR}  (exists: {CODE_DIR.exists()})')
print(f'STORIES_DIR: {STORIES_DIR}  (exists: {STORIES_DIR.exists()})')


CODE_DIR:    C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\uis  (exists: True)
STORIES_DIR: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\stories\uis  (exists: True)


### 2. Locate existing GT story files and their matching generated variants

In [21]:
def find_gt_story_files() -> list[dict]:
    """Returns [{'story_file': Path, 'complexity': str|None, 'index': str}, ...]"""
    results = []

    if TYPE == 'components':
        for complexity in COMPLEXITIES:
            complexity_dir = STORIES_DIR / complexity

            if not complexity_dir.exists():
                print(f'  Skip (not existing): {complexity_dir}')
                continue

            for story_file in sorted(complexity_dir.glob('*.stories.ts')):
                index = story_file.stem.removesuffix('.stories')
                results.append({'story_file': story_file, 'complexity': complexity, 'index': index})

    else:  # 'uis'
        if not STORIES_DIR.exists():
            print(f'  Skip (not existing): {STORIES_DIR}')
        else:
            for story_file in sorted(STORIES_DIR.glob('*.stories.ts')):
                index = story_file.stem.removesuffix('.stories')
                results.append({'story_file': story_file, 'complexity': None, 'index': index})

    return results


def find_generated_variants(complexity: str | None, index: str) -> list[dict]:
    """Returns every generated .vue file matching this GT story's index (and,
    for 'components', its complexity), across all approaches/prompt strategies
    (and, for 'uis', across both pretty/messy variant subfolders).

    Returns: [{'vue_file': Path, 'approach': str, 'prompt_strategy': str,
               'group': str, 'variant': str|None}, ...]
    """
    matches = []

    for approach in APPROACHES:
        for prompt_strategy in PROMPT_STRATEGIES:
            strategy_dir = CODE_DIR / approach / prompt_strategy

            if not strategy_dir.exists():
                continue

            if TYPE == 'components':
                group_dirs = [strategy_dir / complexity] if (strategy_dir / complexity).exists() else []
            else:
                group_dirs = [p for p in strategy_dir.iterdir() if p.is_dir()]  # pretty/, messy/

            for group_dir in group_dirs:
                for vue_file in sorted(group_dir.glob(f'{index}-*.vue')):
                    matches.append({
                        'vue_file': vue_file,
                        'approach': approach,
                        'prompt_strategy': prompt_strategy,
                        'group': group_dir.name,
                        'variant': group_dir.name if TYPE == 'uis' else None,
                    })

    return matches


### 3. Main loop

In [22]:
def to_export_name(approach: str, prompt_strategy: str, variant: str | None, stem: str) -> str:
    """Builds a unique, JS-identifier-safe export name, e.g.
    'b_zero_shot_b1_claude_sonnet_5_1' or, for uis, additionally prefixed
    with the pretty/messy variant."""
    parts = [approach, prompt_strategy]

    if variant:
        parts.append(variant)
    parts.append(stem)

    safe = '_'.join(parts).lower().replace('-', '_')
    safe = re.sub(r'[^a-z0-9_]', '_', safe)

    return safe


def relative_import_path(vue_file: Path, story_file: Path) -> str:
    rel = Path(__import__('os').path.relpath(vue_file, start=story_file.parent)).as_posix()

    return rel if rel.startswith('.') else f'./{rel}'


def inject_generated_stories(story_file: Path, entries: list[dict]) -> None:
    """entries: [{'import_var': str, 'import_path': str, 'export_name': str}, ...]
    Replaces the marker block if present, otherwise appends one."""
    if not story_file.exists():
        raise FileNotFoundError(
            f'Expected an existing GT story file at {story_file}, but it does not exist. '
            f'This script only edits existing stories, it does not create new ones.'
        )

    original = story_file.read_text(encoding='utf-8')

    if BEGIN_MARKER in original:
        pre = original.split(BEGIN_MARKER)[0].rstrip('\n') + '\n\n'
    else:
        pre = original.rstrip('\n') + '\n\n'

    block_lines = [BEGIN_MARKER]

    for e in entries:
        block_lines.append(f"import {e['import_var']} from '{e['import_path']}';")

    block_lines.append('')

    for e in entries:
        block_lines.append(
            f"export const {e['export_name']} = {{ "
            f"render: () => ({{ components: {{ {e['import_var']} }}, "
            f"template: '<{e['import_var']} />' }}) }};"
        )

    block_lines.append(END_MARKER)

    story_file.write_text(pre + '\n'.join(block_lines) + '\n', encoding='utf-8')

In [23]:
gt_stories = find_gt_story_files()
print(f'Found {len(gt_stories)} existing GT story files under {STORIES_DIR}\n')

manifest: list[dict] = []
edited = 0
no_matches: list[str] = []

for gt in gt_stories:
    variants = find_generated_variants(gt['complexity'], gt['index'])

    label = f'{gt["complexity"]}/{gt["index"]}' if gt['complexity'] else gt['index']

    if not variants:
        no_matches.append(label)

        print(f'{label:20s}  0 generated variants found -- skipping (file left untouched)')

        continue

    entries = []
    for v in variants:
        export_name = to_export_name(v['approach'], v['prompt_strategy'], v['variant'], v['vue_file'].stem)
        import_var = 'Comp_' + export_name.removeprefix('s_')  # separate namespace from the export const itself
        import_path = relative_import_path(v['vue_file'], gt['story_file'])

        entries.append({'import_var': import_var, 'import_path': import_path, 'export_name': export_name})

        manifest.append({
            'storyFile': str(gt['story_file'].relative_to(REPO_ROOT)).replace('\\', '/'),
            'importPath': './' + str(gt['story_file'].relative_to(STORYBOOK_ROOT)).replace('\\', '/'),
            'exportName': export_name,
            'approach': v['approach'],
            'prompt_strategy': v['prompt_strategy'],
            'complexity': gt['complexity'],
            'variant': v['variant'],
            'index': gt['index'],
            'stem': v['vue_file'].stem,
            'vuePath': str(v['vue_file'].relative_to(REPO_ROOT)).replace('\\', '/'),
        })

    inject_generated_stories(gt['story_file'], entries)
    edited += 1
    print(f'{label:20s}  {len(variants)} generated variants injected into {gt["story_file"].name}')

print(f'\nEdited {edited} existing story files, {len(manifest)} total generated stories added.')
if no_matches:
    print(f'No generated variants found for: {no_matches}')

Found 5 existing GT story files under C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\stories\uis

1                     108 generated variants injected into 1.stories.ts
2                     108 generated variants injected into 2.stories.ts
3                     108 generated variants injected into 3.stories.ts
4                     108 generated variants injected into 4.stories.ts
5                     108 generated variants injected into 5.stories.ts

Edited 5 existing story files, 540 total generated stories added.


### 4. Save manifest

In [24]:
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Manifest written: {MANIFEST_PATH} ({len(manifest)} entries)')

Manifest written: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\stories_manifest_uis.json (540 entries)
